# 3. Text as Data

**AI and Economics Summer School** — Andrea Ciccarone

The classical text pipeline, end to end:

1. Turn documents into counts
2. Fit a supervised classifier and read which words drive it
3. Fit a topic model and interpret the topics
4. Turn the topics into a variable you could put in a regression

Everything runs on CPU with `scikit-learn`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

plt.rcParams["figure.dpi"] = 120

## 1. Counts, tf-idf, and a supervised classifier

In [ ]:
data = fetch_20newsgroups(subset="all", categories=["talk.politics.guns", "sci.med"],
                          remove=("headers", "footers", "quotes"), random_state=0)

# fetch_20newsgroups sorts the categories, so read the order off the object
# rather than assuming it matches what you asked for.
names = data.target_names
print("class 0 =", names[0], "| class 1 =", names[1])

# Real corpora contain junk: uuencoded binaries, vote tallies, signature blocks.
# One cheap filter goes a long way, and you should always look at what you dropped.
def looks_like_text(d):
    if not (200 < len(d) < 6000):
        return False
    longest_token = max((len(t) for t in d.split()), default=0)
    return longest_token < 40          # uuencoded blobs have enormous "words"

keep = [i for i, d in enumerate(data.data) if looks_like_text(d)]
docs = [data.data[i] for i in keep]
y = data.target[keep]
print(f"kept {len(docs)} of {len(data.data)} documents")

In [ ]:
vec = TfidfVectorizer(stop_words="english", min_df=5, max_df=0.5,
                      ngram_range=(1, 1), sublinear_tf=True)

X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(docs, y, test_size=0.3,
                                                  stratify=y, random_state=0)

# Fit the vectorizer on TRAINING DATA ONLY. Fitting on everything leaks.
X_tr = vec.fit_transform(X_tr_raw)
X_te = vec.transform(X_te_raw)
print("vocabulary:", len(vec.vocabulary_), "| training matrix:", X_tr.shape)

clf = LogisticRegression(max_iter=2000, C=1.0).fit(X_tr, y_tr)
print("test AUC:", round(roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1]), 3))

> **The most common leak in applied text work** is fitting the vectorizer (or a
> scaler, or a feature selector) on the full sample before splitting. The
> vocabulary and the idf weights are estimated quantities. They belong inside the
> training fold.

In [ ]:
# Which words drive the classification? This is why we kept counts rather than embeddings.
terms = np.array(vec.get_feature_names_out())
coef = clf.coef_[0]
top = np.argsort(coef)
# Negative coefficients push toward class 0, positive toward class 1.
print(f"Most predictive of '{names[0]}' (class 0):")
print("  ", ", ".join(terms[top[:15]]))
print(f"\nMost predictive of '{names[1]}' (class 1):")
print("  ", ", ".join(terms[top[-15:]][::-1]))

## 2. A topic model

LDA on counts (not tf-idf: the generative model is about counts).

In [ ]:
cnt_vec = CountVectorizer(stop_words="english", min_df=10, max_df=0.4)
C = cnt_vec.fit_transform(docs)
vocab = np.array(cnt_vec.get_feature_names_out())

lda = LatentDirichletAllocation(n_components=6, learning_method="batch",
                                max_iter=25, random_state=0).fit(C)

for k, comp in enumerate(lda.components_):
    print(f"Topic {k}: " + ", ".join(vocab[np.argsort(-comp)[:10]]))

In [ ]:
theta = lda.transform(C)      # document-topic proportions
print("theta shape:", theta.shape, "| rows sum to 1:", np.allclose(theta.sum(1), 1))

df = pd.DataFrame(theta, columns=[f"topic{k}" for k in range(theta.shape[1])])
df["group"] = [names[i] for i in y]
display(df.groupby("group").mean().round(3))

`theta` is the payoff. Each document is now six numbers instead of a wall of text,
and those numbers are interpretable. **This is the variable you would put in a
regression.**

Read the table above: the topic shares differ across the two groups, which is the
kind of variation you would go on to explain.

In [ ]:
# Read the documents that load most heavily on each topic. Always do this before
# naming a topic.
for k in range(lda.n_components):
    top_doc = np.argsort(-theta[:, k])[0]
    print("=" * 70)
    print(f"TOPIC {k}: " + ", ".join(vocab[np.argsort(-lda.components_[k])[:8]]))
    print("  most representative document:")
    print("   ", docs[top_doc][:220].replace("\n", " "), "...")

## Exercises

1. Swap the two newsgroups for a harder pair (e.g. `sci.med` vs `sci.space`).
   What happens to the AUC, and which words now carry the signal?
2. Change `ngram_range=(1, 1)` to `(1, 2)` so bigrams are counted. Does the
   classifier improve? Look at the top features again.
3. Refit LDA with 3 topics and with 12. At which number can you still name every
   topic from its top words and its representative document?
4. Compare the two representations directly: fit the same logistic regression on
   the 6 LDA topic proportions instead of the full tf-idf matrix. How much AUC do
   you lose by compressing 20,000 features into 6?